# Flask + Jinja + HTML: les templates

On sait maintenant:
- comment créer une application Flask 
- comment créer des routes, et des routes à paramètres
- comment afficher du texte dans le navigateur

**Mais**: pour le moment, notre site est MOCHE, et, plus grave, il n'utilise pas du tout de HTML, qui est quand même la base du web. On est assez loin d'avoir fait le nouveau Gallica. Aujourd'hui, on a donc apprendre à **générer du HTML en Python dans une appli Flask, via la librairie Jinja**.

---

# Flask + HTML

Pour le moment, nos fonctions ne renvoient que du texte:

In [2]:
from flask import Flask

app = Flask("Ma première appli")

@app.route("/")
def index():
    return "Hello world !"

Pour afficher du HTML, on peut **créer des vues qui retournent du HTML au lieu du texte**, mais ça va vite devenir douloureux à écrire et très difficile à maintenir. Donc, on ne verra qu'une fois comment faire. 

> **Dans votre terminal, après avoir sourcé l'env, lancer l'appli**
> ```bash
> python ./apps/s2/html/main.py
> ```

In [8]:
app = Flask("templates HTML")

@app.route("/")
def index():
    app_name = "Catalogue Richelieu"
    html = f"""
        <html>
            <head>
                <title>{app_name}</title>
            </head>
            <body>
                <h1>Bienvenue sur le {app_name} !</h1>
            </body>
        </html>
    """
    return html

Pour monter une vraie appli, on va avoir besoin d'un système beaucoup plus complet pour générer dynamiquement du HTML à partir de Python, et c'est ici que les templates rentrent en jeu.

---

# Templates Jinja: une introduction

## Jinja ?

**[Jinja](https://jinja.palletsprojects.com/en/stable/) est un "moteur de templates"** (*template engine*): une librairie qui, à partir de fichiers templates, génère dynamiquement du HTML adapté à vos données et votre code Python. **Parmi les opérations possibles**:
- remplacer des variables Python par leurs valeurs
- boucler sur des variables Python pour créer des listes
- gérer des structures conditionnelles (`if`/`else`), pour n'afficher du contenu que si certaines conditions sont réunies
- combiner ensemble plusieurs templates
- et plein d'autres choses !

Jinja est une librairie développée par [Pallets projects](https://palletsprojects.com/), qui sont les créateurs de Flask. L'intégration Flask/Jinja est donc très transparente. À noter:
- Jinja peut être utilisé sans Flask
- Jinja peut être utilisé pour générer n'importe quel type de fichier texte: HTML, XML, Latex...

## Utiliser des templates

Une première application utilisant Jinja se trouve dans `./apps/s2/jinja_base/main.py`.

> **Lancer l'appli**
> ```bash
> python ./apps/s2/jinja_base/main.py
> ```

### Code

Pour que notre vue retourne une template Jinja plutôt qu'une chaîne de caractère, il faut juste utiliser la fonction `render_template`:

In [6]:
from flask import render_template

app_name = "Catalogue Richelieu"

app = Flask(app_name)

@app.route("/")
def index():
    return render_template("homepage.html", app_name=app_name)

### Syntaxe

#### Côté python

Pour utiliser `render_template`, 
- on **importe la fonction**
- on **fait une vue qui retourne le résultat de `render_template`**. `render_template` prend pour arguments:
    - en 1er argument **le nom d'une template Jinja**: `homepage.html`
    - ensuite, **les valeurs à rendre accessibles dans le template**: `app_name=app_name`. La syntaxe est: `nom_de_variable_dans_la_template=nom_de_variable_en_python`.

#### Côté Jinja

Voici `homepage.html`:

```html
<html>
    <head>
        <title>{{app_name}}</title>
    </head>
    <body>
        <h1>Bienvenue sur le {{app_name}} !</h1>
    </body>
</html>
```

On voit que c'est un fichier HTML, **mais**, qui contient `{{app_name}}`: 
- les noms de variables Python sont **encadrés de doubles accolades (`{{}}`)**
- `render_template` **remplacera le nom de variable** par sa valeur: c'est de *l'interpolation de variables*

---

# Bonnes pratiques: structurer le code d'une application Flask

Jusqu'à maintenant, toutes nos applis étaient contenues dans un seul fichier. Maintenant, on arrive à **une appli organisée en module** (un dossier avec plusieurs sous-dossiers). Voilà comment l'appli est organisée, et voilà comment toutes nos applis seront plus où moins organisées pour le reste de nos cours:

```txt
├── main.py ...............: fichier qui lance l'appli
└──app ....................: dossier contenant notre application
   ├── app.py .............: fichier qui définit notre `app` Flask 
   ├── __init__.py
   ├── routes .............: dossier qui contient toutes nos routes Flask
   │   ├── generic.py .....: pour le moment, toutes nos routes seront contenues dans ce fichier 
   │   └── __init__.py
   ├── templates ..........: dossier contenant nos templates HTML
   │   └── homepage.html ..: la template affichée juste au dessus
   └── utils ..............: code utilitaire
       └── constants.py ...: toutes les constantes
```

Pour rappel, `__init__.py` transforme un dossier en module et permet de faire des imports entre fichiers de ce module.

Regardons dans le détail, en commençant par les petits blocs.

## `app/templates/`

Ce dossier contient **toutes nos templates Jinja HTML**.

## `app/utils/` et `constants.py`

**Le dossier `utils/` stocke les "utilitaires"**, c'est à dire les petites fonctions qui n'ont pas vraiment d'autres endroits où aller et qui sont utilisées partout dans l'appli (par ex.: fonctions qui gèrent la lecture/écriture de fichiers). Idéalement, `utils/` ne doit pas contenir des éléments "sensibles" (comme des opérations sur base de données).

On y met `constants.py` qui contient toutes les *constantes* (variables définies au lancement de l'appli qui seront pas modifiées ensuite). Ces constantes sont nos chemins vers des fichiers/dossiers importants:

```py
# chemin absolu vers notre dossier de templates (app/templates/)
DIR_TEMPLATES = DIR_APP / "templates" 
```

## `app/routes/`et `generic.py`

**Le dossier `routes/` va contenir toutes les routes** de notre appli. Pour l'instant, toutes nos routes sont dans un fichier (`generic.py`), mais si notre appli grandit c'est une bonne idée de les scinder en plusieurs fichiers.

Voilà le contenu de `generic.py`:

```py
from flask import render_template

# note: app.app = app/app.py => on importe la variable `app` du fichier `app/app.py`
from app.app import app

@app.route("/")
def index():
    app_name = "Catalogue Richelieu"
    return render_template("homepage.html", app_name=app_name)
```

On voit que:
- **on importe `app`**, définie dans `app/app.py` pour pouvoir utiliser `@app.route`
- pour le reste, **le code est inchangé**

### `app/app.py`

**C'est le ficher qui définit notre appli Flask**. Voilà son contenu:

```py
from flask import Flask

# on importe le chemin vers `templates/` qui est dans `app/utils/constants`
from app.utils.constants import DIR_TEMPLATES

# on définit notre appli
app = Flask(
    "Jinja base",
    template_folder=DIR_TEMPLATES
)

# on importe les routes pour qu'elles soient utilisables dans l'application.
from app.routes import generic
```

À noter:
- `template_folder=DIR_TEMPLATES` permet d'indiquer à Flask **où chercher les templates Jinja**: dans notre dossier `templates/`. 
    - => **tous les chemins vers des templates dans `render_template`** seront définis relativement au dossier `templates/`.
    - à noter que dans certaines configurations, `template_folder` est optionnel.
- `from app.routes import generic` permet **d'importer les routes** pour qu'elles soient utilisables.
    - d'habitude, les imports sont faits au début du fichier.
    - ici, c'est important **d'importer routes.generic à la fin du fichier pour éviter un import circulaire**. sinon, 
        ```txt
        1. `app/routes/generic.py`                                
           exécute  →  from app import app                        
        2. python doit résoudre le module `app`                   
           →  il commence à exécuter `app/app.py`                 
        3. or `app/app.py` exécute lui-même:                      
           →  `import app.routes.generic`                         
        4. python revient donc dans `app/routes/generic.py`,      
           → python doit résoudre le module `app`                 
                                                                  
        → import circulaire: boucle infinie !                     
        ``` 

## `main.py`

**C'est le fichier qui lance notre appli Flask** et le fichier qu'on éxécute avec Python:

```py
# on importe la variable `app` de `app/app.py`
from app.app import app

if __name__ == "__main__":
    app.run(debug=True)
```

---

# Templates Jinja: la syntaxe

Une template doit être lue commme un fichier HTML classique, jusqu'à ce qu'on arrive sur du code Jinja.

La documentation offcielle [est ici](https://jinja.palletsprojects.com/en/stable/templates/).

## Interpolation de variables

On l'a vu, interpoler une variable, c'est **remplacer son nom par sa valeur**.

> **Relancer l'appli précédente:**
> ```bash
> python ./apps/s2/jinja_base/main.py
> ```

**Pour résumer**:
- **dans la template**, on indique la variable à remplacer avec `{{}}`: `{{nom_de_variable}}`
    ```html
    <h1>Bienvenue sur le {{app_name}} !</h1>
    ```
- **dans `render_template`**, on passe une valeur pour la variable à interpoler:
    ```py
    render_template("homepage.html", app_name=app_name)
    ```

## Manipuler des listes

L'indexation dans une template Jinja marche **exactement comme en Python: `{{ma_liste[mon_index]}}`** en Jinja est équivalent à `ma_liste[mon_index]` en Python. Par exemple, `{{ma_liste[0]}}` retourne le 1er élément de la liste.

> **Lancer l'appli `jinja_list`**:
> ```bash
> python ./apps/s2/jinja_list/main.py
> ```

**Dans notre appli Flask**, on ajoute quelques ressources iconographiques: 
```py
# on ajoute quelques ressources iconographiques à notre site et on passe `icono` à `render_template`:
icono = [
    "Palais Royal (1909-1927)",
    "Palais Royal (1857-1870)",
    "Hôtel de la Chancellerie d'Orléans : 19 rue des Bons Enfants",
    "Omnibus tiré par des chevaux, près du 9 rue de Valois, lieu de travail du photographe Zulimo Chiesi",
    "Groupe des délégués internationaux au 35e congrès de l'Union des sociétés de gymnastique de France",
]

# puis on passe `icono` à `render_template`. `icono` devient maintenant une variable utilisable côté Jinja
@app.route("/")
def index():
    return render_template("homepage.html", app_name=app_name, icono=icono)
```

**Côté Jinja**, on modifie `homepage.html` en utilisant `{{icono[0]}}` et `{{icono[-1]}}` pour voir comment marche l'indexation:

```html
<p>Le titre de la première ressource est:
    <b>{{icono[0]}}.</b></p>

<p>Le titre de la dernière ressource est:
    <b>{{icono[-1]}}.</b></p>
```

## Manipuler des dictionnaires

En Jinja, il y a deux manières d'accéder aux valeurs d'un dictionnaire:
- comme en Python: `{{mon_dict[ma_cle]}}`
- en remplaçant les crochets par un point `.`: `{{mon_dict.ma_cle}}`

> **Lancer l'appli `jinja_dict`**:
> ```bash
> python ./apps/s2/jinja_dict/main.py
> ```

**Côté Python**, pour cet exemple **et jusqu'à la fin de cette séance**, `icono` sera une liste de dictionnaires:

```py
icono = [
    {
        "id": 0,
        "auteurice": "Eugène Atget",
        "titre": "Palais Royal",
        "date": "1909-1927",
        "url": "https://quartier-richelieu.inha.fr/iconographie/qr12ad70f104f8245b69df7d1b35ac70b9f",
    },
    {
        "id": 1,
        "auteurice": "Jules Arnoult",
        "titre": "Palais Royal",
        "date": "1857-1870",
        "url": "https://quartier-richelieu.inha.fr/iconographie/qr12e06bf9e7d31433f9c3f266e8970404d",
    },
    ...
]
```

**Côté Jinja**, on voit que on peut facilement manipuler des listes contenant des dictionnaires:

```html
<p>Voici la fiche de la première ressource iconographique:</p>
<ul>
    <li>Auteur.ice: {{icono[0]["auteurice"]}}</li>
    <li>Titre: {{icono[0]["titre"]}}</li>
    {# ici, on utilise `dict.cle` et non `dict["cle"]` #}
    <li>Date: {{icono[0].date}}</li>
    <li>URL source: 
        <a href="{{icono[0].url}}">
            {{icono[0].url}}
        </a>
    </li>
</ul>
```

## Filtres

En Jinja, comme en Python, on peut manipuler des variables. Mais, les fonctions Python, comme `len()` ne fonctionnent pas. Il faut **utiliser des filtres**.

Un filtre permet de **modifier la manière dont le contenu d'une variable sera affiché** par Jinja. 
- ce sont des **équivalents à des méthodes Python** comme `"texte".upper()`.
- la syntaxe est: `{{ ma_variable|mon_filtre }}`. `|` veut dire qu'on applique `mon_filtre` à `ma_variable`.

> **Lancer l'appli `jinja_filtre`**
> ```bash
> python ./apps/s2/jinja_filtre/main.py
> ```

On va voir un seul filtre: **`length`**. Il est est **l'équivalent Jinja de `len()`.**

**Dans notre template Jinja**, on modifie `homepage.html` pour y ajouter le nombre de ressources iconographiques sur notre site. Pour ça, **on utilise `{{icono|length}}`** !

```html
<p>Il y a <b>{{icono|length}} ressources iconographiques</b> 
    sur ce site.</p>
```

Il existe beaucoup de filtres Jinja: voici [la liste complète](https://jinja.palletsprojects.com/en/stable/templates/#builtin-filters). On ne le verra pas, mais vous pouvez même créer des filtres personnalisés via des fonctions Python !

## Les structures conditionnelles `if`/`elif`/`else`

Dans un site, on ne va vouloir afficher des informations que si certaines conditions sont réunies. Dans notre cas, on veut faire un "focus sur Eugène Atget", mais ça n'a de sens que si on a des photographies de lui à afficher.

> **Lancer l'appli `jinja_if`**
> ```bash
> python ./apps/s2/jinja_if/main.py
> ```

### Code

**Côté Flask**, on modifie notre route pour compter le nombre d'oeuvres d'Atget dans `icono`:

```py
@app.route("/")
def index():
    app_name = "Catalogue Richelieu"
    
    # on récupère le nombre d'oeuvres d'Eugène Atget
    atget_count = 0
    for item in icono:
        if item["auteurice"] == "Eugène Atget":
            atget_count += 1

    return render_template("homepage.html", app_name=app_name, icono=icono, atget_count=atget_count)
```

**On ajoute un `if` à notre template Jinja**:
```html
{% if atget_count > 0 %} 
    <p>Il y a <b>{{ atget_count }} oeuvres d'Atget</b> dans notre corpus (quelle chance !).</p>
{% endif %}
```

### Syntaxe

**Pour résumer, pour faire un affichage conditionnel en Jinja**, 
- **on utilise `{% if %}`** (attention à ne pas oublier le `{% endif %}` !):
    ```html
    {% if condition %}
        ...
    {% endif %}
    ```
- on peut multiplier les conditions avec **`{% elif %}` et `{% else %}`**:
    ```html
    {% if condition1 %}
        Ce bloc ne s'affichera que si `condition1` est `True`
    {% elif condition2 %}
        Ce bloc ne s'affichera que si `condition2` est `True`
    {% else %}
        Ce bloc s'affichera si `condition1` et `condition2` sont `False`
    {% endif %}
    ```

## Les boucles `for`

On peut faire des structures conditionnelles en Jinja, c'est donc logique qu'on fasse aussi des `for` ! Et c'est bien pratique puisque notre catalogue contient énormément de structures qui se ressemblent.

> **Lancer l'appli `jinja_for`**
> ```bash
> python ./apps/s2/jinja_for/main.py
> ```

### Le code

Dans notre cas, on utilise `for` pour **montrer les mêmes informations** sur chacune de nos ressources iconographiques:

```html
<ul>
    {% for icono_item in icono %}
        <li>Item {{ icono_item.id }}:
            <ul>
                <li>Auteur.ice: {{ icono_item.auteurice }}</li>
                <li>Titre: {{ icono_item.titre }}</li>
                {# ici, on utilise `dict.cle` et non `dict["cle"]` #}
                <li>Date: {{ icono_item.date }}</li>
                <li>URL source: 
                    <a href="{{ icono_item.url }}">
                        {{ icono_item.url }}
                    </a>
                </li>
            </ul>
        </li>
    {% endfor %}
</ul>
```

### Syntaxe

**La syntaxe** est très proche de Python (**attention à ne pas oublier le `endfor` !**): 
```html
{% for item in iterable %}
    {{ item }}
{% endfor %}
```

## Les liens entre les pages: `url_for`

Notre page d'accueil commence à être bien chargée. On **modifie donc le site**:
- la page d'accueil contient un index des ressources iconographiques
- chaque ressource iconographique a sa propre page.

> **Lancer l'appli `jinja_url_for`**
> ```bash
> python ./apps/s2/jinja_url_for/main.py
> ```

### Le code

**En Python, on ajoute une route**: la vue principale de chaque ressource iconographique. Cette fonction:
- prend en argument `id_icono`.
- se sert de `id_icono` pour sélectionner `item_icono`, l'item principal dont on veut des informations
- render la template `icono_main` en lui passant `item_icono`. 

```python
@app.route("/iconographie/<int:id_icono>")
def icono_main(id_icono: int):
    for item in icono:
        if item["id"] == id_icono:
            item_icono = item
    return render_template("icono_main.html", item_icono=item_icono)
```

**Côté Jinja,**
- **on a deux pages HTML**: en plus de `homepage.html`, on ajoute `icono_main.html`.
    ```txt
    templates/
    ├── homepage.html
    └── icono_main.html
    ```
- `icono_main.html` affiche les **informations sur une ressource**:
    ```html
    <ul>
        <li>Auteur.ice: {{ item_icono.auteurice }}</li>
        <li>Titre: {{ item_icono.titre }}</li>
        {# ici, on utilise `dict.cle` et non `dict["cle"]` #}
        <li>Date: {{ item_icono.date }}</li>
        <li>URL source: 
            <a href="{{ item_icono.url }}">
                {{ item_icono.url }}
            </a>
        </li>
    </ul>
    ```
- `homepage.html` permet, **pour chaque item d'`icono`, de naviguer** sur la page principale de cet item:
    ```html
    {% for item in icono %}
        <li>
            Item {{ item.id }}: 
            <a href="{{ url_for('icono_main', id_icono=item.id) }}">{{ item.titre }}</a>
        </li>
    {% endfor %}        
    ```

### Syntaxe d'`url_for`

**Pour naviguer d'une page à l'autre, on a utilisé `url_for`**.
- en 1er argument, **`url_for` prend le nom de la vue** vers laquelle on veut naviguer: `icono_main`.
- les autres arguments sont des **paramètres d'URL**: `id_icono=item.id`.
- en langage humain, `url_for` *redirige vers l'URL associé à la fonction `icono_main` et donne à l'argument `id_icono` la valeur de `item.id`*.
- **attention**: `url_for` ne peut rediriger que vers des vues (définies avec `@app.route()`)

**La syntaxe est donc**:

```py
url_for('nom_de_fonction', nom_d'argument=valeur)
```

## Combiner des templates: `includes`/`extends`



# TLDR

code | usage
-----|------
`render_template` | utiliser une template Jinja en Flask
`url_for(ma_vue)` | rediriger vers la vue `ma_vue` 
`{# ... #}` | commentaire en Jinja
`{{ ma_variable }}` | afficher le contenu de `ma_variable` en Jinja
`{{ ma_variable\|length }}` | afficher la longueur de `ma_variable` en Jinja
`{{ ma_variable[i] }}` | si `ma_variable` est indexée (est une `str` ou `list`), afficher l'item à la position `i` (avec `i` nombre entier)
`{{ ma_variable["ma_cle"] }}` | si `ma_variable` est un `dict`, afficher la valeur de `ma_variable["ma_cle"]`
`{{ ma_variable.ma_cle. }}` | pareil
`{% if condition %}...{% endif %}` | affichage conditionel Jinja
`{% for item in iterable %}...{% endfor %}` | boucler sur un itérable (`list`) en Jinja  

---
---

 programme des hostilités:
- HTML basique
- HTML avec interpolation de variables, et pourquoi c relou
- templates Jinja:
    - `render_template`
    - syntaxe: interpolation de variables
    - synxate: for
    - syntaxe: if/then/else
    - syntaxe: include/extends (où mettre ça ?)
    - syntaxe: les filtres et la manipulation de variables
    - un brin de CSS
- la gestion d'erreurs